# 02 - Microstructure Noise Modeling and Logit Conversion

Goal: model heteroskedastic observation noise and transform probabilities to log-odds.

Formulas:
$$
y_t = peratorname{logit}(p_t) = ogeft(rac{p_t}{1-p_t}
ight) = x_t + arepsilon_t
$$
$$
igma^2_{arepsilon,t} = a_0 + a_1s_t^2 + a_2d_t^{-1} + a_3r_t + a_4ota_t^2
$$
where $s_t$ is spread, $d_t$ depth proxy, $r_t$ trade-rate proxy, and $ota_t$ imbalance proxy.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
inp = Path('stage1_preprocessed.csv')
if not inp.exists():
    raise FileNotFoundError('Run notebook 01 first to generate stage1_preprocessed.csv')

df = pd.read_csv(inp)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
df = df.sort_values('timestamp').reset_index(drop=True)
df.head()

,timestamp,bid,ask,trade_size,spread,p_clipped
0,2025-11-14 16:31:16+00:00,0.955,0.99999,0.0,0.04499,0.977495


## Build feature proxies for heteroskedastic noise

In [3]:
eps = 1e-8
df['spread'] = pd.to_numeric(df['spread'], errors='coerce').fillna(df['spread'].median())
df['trade_size'] = pd.to_numeric(df['trade_size'], errors='coerce').fillna(0.0)
df['p_clipped'] = pd.to_numeric(df['p_clipped'], errors='coerce').clip(1e-5, 1 - 1e-5)

# Depth proxy from inverse spread
df['depth_proxy'] = 1.0 / (df['spread'] + eps)

# Trade-rate proxy from local rolling trade flow
df['trade_rate_proxy'] = df['trade_size'].rolling(30, min_periods=1).mean()

# Imbalance proxy from signed probability changes weighted by size
dp = df['p_clipped'].diff().fillna(0.0)
signed_flow = np.sign(dp) * df['trade_size']
num = signed_flow.rolling(30, min_periods=1).sum()
den = df['trade_size'].rolling(30, min_periods=1).sum().replace(0.0, np.nan)
df['imbalance_proxy'] = (num / den).fillna(0.0).clip(-1, 1)

df[['spread', 'depth_proxy', 'trade_rate_proxy', 'imbalance_proxy']].describe().T

,count,mean,std,min,25%,50%,75%,max
spread,1.0,0.044990,NaN,0.044990,0.044990,0.044990,0.044990,0.044990
depth_proxy,1.0,22.227157,NaN,22.227157,22.227157,22.227157,22.227157,22.227157
trade_rate_proxy,1.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000
imbalance_proxy,1.0,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000


## Calibrate a practical variance model and apply logit transform

In [4]:
# Target proxy: short-horizon squared innovations in probability space
innov_proxy = df['p_clipped'].diff().fillna(0.0) ** 2

X = np.column_stack([
    np.ones(len(df)),
    (df['spread'] ** 2).values,
    (1.0 / (df['depth_proxy'] + 1e-12)).values,
    df['trade_rate_proxy'].values,
    (df['imbalance_proxy'] ** 2).values,
])
y = innov_proxy.values

# OLS coefficients for demo purposes
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
sigma2_eps = X @ beta
sigma2_eps = np.clip(sigma2_eps, np.quantile(sigma2_eps, 0.02), np.quantile(sigma2_eps, 0.98))

df['sigma2_eps'] = sigma2_eps
df['y_logit'] = np.log(df['p_clipped'] / (1.0 - df['p_clipped']))

print('beta:', beta)
df[['timestamp', 'p_clipped', 'y_logit', 'sigma2_eps']].head()

beta: [-0.  0.  0.  0.  0.]


,timestamp,p_clipped,y_logit,sigma2_eps
0,2025-11-14 16:31:16+00:00,0.977495,3.771256,0.0


In [5]:
out_cols = [
    'timestamp', 'bid', 'ask', 'trade_size', 'spread', 'p_clipped',
    'depth_proxy', 'trade_rate_proxy', 'imbalance_proxy',
    'sigma2_eps', 'y_logit',
]
df[out_cols].to_csv('stage2_logit_noise.csv', index=False)
print('saved:', Path('stage2_logit_noise.csv').resolve())
print('rows:', len(df))

saved: C:\Users\p\Documents\GitHub\volatility-estimator\research\stage2_logit_noise.csv
rows: 1
